<a href="https://colab.research.google.com/github/medijay/msc-ai-phishing-detection/blob/main/notebooks/03_bert_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install transformers datasets torch scikit-learn

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/Msc_phishing_Project/datasets/clean_allam_dataset.csv")

# ✅ CLEAN DATA FIRST (VERY IMPORTANT)
df = df.dropna(subset=['clean_email'])
df['clean_email'] = df['clean_email'].astype(str)

# Prepare data
X = df['clean_email']
y = df['label']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization function
def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

# ✅ TOKENIZE (OUTSIDE FUNCTION)
train_encodings = tokenize_data(X_train)
test_encodings = tokenize_data(X_test)

# Dataset class
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create datasets
train_dataset = EmailDataset(train_encodings, y_train)
test_dataset = EmailDataset(test_encodings, y_test)

# Load model
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,  # keep 1 for speed
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=100
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train
trainer.train()

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

print("BERT Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

BERT Accuracy: 0.9893313935867127
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      8010
           1       0.99      0.99      0.99      8487

    accuracy                           0.99     16497
   macro avg       0.99      0.99      0.99     16497
weighted avg       0.99      0.99      0.99     16497

